# Ray of Hope: ansiedad y depresión en el mundo

Este cuaderno reúne datos de salud mental y contexto socioeconómico para **2014–2023**, la última década completa disponible en los datos GBD consultados. Conserva un registro por país y año para estudiar las tendencias.

**Pregunta:** ¿Qué países mantienen las prevalencias estimadas más altas de ansiedad y depresión durante la década, y qué contexto socioeconómico presentan?

**Fuentes:** exportación CSV del [IHME Global Burden of Disease Results Tool](https://vizhub.healthdata.org/gbd-results/) y API de Indicadores del [Banco Mundial](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392).

**Por qué hay un CSV local para IHME:** la descarga programática de la gráfica de Our World in Data devuelve 403 para estos datos, y la descarga de GBD se hace desde la herramienta oficial. No automatizamos ni eludimos ese acceso: descarga el CSV autorizado y guárdalo en la ruta indicada abajo.


## 1. Obtener el archivo de prevalencia de IHME

La herramienta de resultados de IHME requiere que el usuario inicie sesión para descargar archivos. En [GBD Results Tool](https://vizhub.healthdata.org/gbd-results/), genera y descarga un CSV con estas selecciones:

- **Medida:** Prevalence
- **Causas:** Anxiety disorders y Depressive disorders
- **Sexo:** Both
- **Edad:** All ages y Age-standardized
- **Métrica:** Number y Rate
- **Años:** 2014–2023
- **Ubicaciones:** países / todas las ubicaciones disponibles

Guárdalo como `data/raw/gbd_mental_health_2014_2023.csv` dentro de la carpeta del proyecto. El export debe incluir dimensiones equivalentes a `location_name`, `year`, `cause_name`, `measure_name`, `metric_name`, `age_name`, `sex_name` y `val`.

La función limpia el archivo y empareja países con códigos ISO3 del Banco Mundial, usando equivalencias de nombres cuando hacen falta. Revisa luego los países que queden sin emparejar.


## 2. Preparación

Instala las dependencias una vez desde la terminal con `pip install -r requirements.txt`. Mantén `ray_of_hope_data.py` junto a este cuaderno. Necesitarás internet para consultar la API del Banco Mundial y el CSV de IHME guardado localmente.


In [3]:
# Preparar archivos en Google Colab o en Jupyter local
import os
import sys
from pathlib import Path

if Path("/content").exists() and not Path("ray_of_hope_data.py").exists():
    from google.colab import files
    from io import BytesIO
    import zipfile

    print("Sube ray-of-hope-project.zip")
    uploaded_zip = files.upload()
    zip_name = next(name for name in uploaded_zip if name.lower().endswith(".zip"))
    with zipfile.ZipFile(BytesIO(uploaded_zip[zip_name])) as archive:
        archive.extractall("/content")
    os.chdir("/content/ray-of-hope-project")

    raw_dir = Path("data/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)
    expected_gbd = raw_dir / "gbd_mental_health_2014_2023.csv"
    if not expected_gbd.exists():
        print("Ahora sube el CSV descargado desde IHME GBD Results Tool")
        uploaded_gbd = files.upload()
        gbd_name = next(iter(uploaded_gbd))
        expected_gbd.write_bytes(uploaded_gbd[gbd_name])
else:
    print("Carpeta del proyecto detectada:", Path.cwd())

sys.path.insert(0, str(Path.cwd()))


Carpeta del proyecto detectada: c:\Users\Ariane\Downloads


In [5]:
import pandas as pd
from IPython.display import display
from ray_of_hope_data import (
    START_YEAR, END_YEAR, MIN_YEARS_FOR_RANKING,
    build_dataframes, rank_top10,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"Periodo: {START_YEAR}–{END_YEAR}; mínimo de años para el ranking: {MIN_YEARS_FOR_RANKING}")


ModuleNotFoundError: No module named 'ray_of_hope_data'

## 3. Extracción, limpieza y combinación

Se consultan tres indicadores de la API del Banco Mundial: población, PIB per cápita y grupo de ingresos actual. Se combinan con el CSV autorizado de IHME por país (ISO3) y año.


In [4]:
GBD_CSV = "data/raw/gbd_mental_health_2014_2023.csv"
df_mental, df_world_bank, df_decada = build_dataframes(GBD_CSV)
print("Filas IHME limpiadas:", len(df_mental))
print("Filas del Banco Mundial:", len(df_world_bank))
print("Filas combinadas país-año:", len(df_decada))
print("Años presentes:", sorted(df_decada["year"].dropna().unique().tolist()))
display(df_decada.head(10))


NameError: name 'build_dataframes' is not defined

## 4. Revisar calidad y cobertura

Comprobamos que solo haya una fila por país y año, y vemos cuántos años de prevalencia están disponibles para cada trastorno.


In [ ]:
print("Duplicados por país-año:", df_decada.duplicated(["iso3", "year"]).sum())
missing = (
    df_decada[["anxiety_age_standardized_pct", "depression_age_standardized_pct",
               "anxiety_estimated_cases", "depression_estimated_cases",
               "population", "gdp_per_capita_usd", "income_group"]]
    .isna().mean().mul(100).round(1).rename("missing_pct").to_frame()
)
display(missing)


## 5. Top 10 por media de la década

Los rankings usan la **prevalencia media ajustada por edad** durante 2014–2023, que permite comparar países con estructuras de edad distintas. Se muestran por separado ansiedad y depresión. El conteo estimado usa la métrica Number del GBD, no la tasa ajustada por edad.


In [ ]:
df_resumen, top10_ansiedad, top10_depresion = rank_top10(df_decada)
cols_ansiedad = ["country", "iso3", "anxiety_years_available", "anxiety_mean_pct",
                 "anxiety_cases_mean", "gdp_per_capita_latest_year_usd",
                 "current_world_bank_income_group"]
cols_depresion = ["country", "iso3", "depression_years_available", "depression_mean_pct",
                  "depression_cases_mean", "gdp_per_capita_latest_year_usd",
                  "current_world_bank_income_group"]
print("Top 10 — ansiedad")
display(top10_ansiedad[[c for c in cols_ansiedad if c in top10_ansiedad.columns]])
print("Top 10 — depresión")
display(top10_depresion[[c for c in cols_depresion if c in top10_depresion.columns]])


## 6. Tendencias anuales

El promedio por año que sigue es un promedio simple de los países con datos; no está ponderado por población ni debe presentarse como prevalencia mundial oficial.


In [ ]:
tendencia = (
    df_decada.groupby("year", as_index=False)[
        ["anxiety_age_standardized_pct", "depression_age_standardized_pct"]
    ].mean().round(3)
)
display(tendencia)


## 7. Exportar el dataframe limpio



In [ ]:
from pathlib import Path
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / f"df_salud_mental_{START_YEAR}_{END_YEAR}.csv
df_decada.to_csv(csv_path, index=False, encoding="utf-8")
print(f"CSV guardado en: {csv_path.resolve()}")


## Notas metodológicas

- La tasa de prevalencia ajustada por edad se usa para comparar; no equivale a la tasa cruda de la población.
- La métrica Number del GBD ofrece una estimación de casos para cada país y año. Es una estimación modelada, no un conteo de personas diagnosticadas.
- El PIB per cápita está expresado en USD corrientes y se muestra como contexto, no como medida directa de vulnerabilidad individual.
- El grupo de ingresos es la clasificación actual publicada por el Banco Mundial; no es una serie histórica anual.
- El top 10 exige 8 años disponibles como mínimo para cada trastorno. Comparen la cobertura antes de interpretar diferencias.
- No se incluyen agregados regionales en la lista de países.
